# Q-TRACE hardware experiment: multiplexed BB84 with physical eavesdroppers on IBM Quantum

This notebook runs the hardware part of Q-TRACE v2 (Sec. 5 of the paper). Up to 40 independent BB84
links run in parallel on one IBM Heron processor; each link is a (data, ancilla) qubit pair. During
monitoring epochs some links are attacked by an eavesdropper that is **physically executed** as a
two-qubit interaction with her ancilla:

* `probe(θ)`: `RZX(θ)` or `RXX(θ)` entangling probe, ancilla read in the Helstrom-optimal Y basis
  (θ = π/2 is intercept–resend);
* `ad(γ)`: amplitude-damping coupling that produces bit-asymmetric errors.

**QPU budget.** A pilot job (8 circuits × 400 shots) measures the actual quantum-seconds per shot on
your account; the main job's shots are then chosen so that pilot + main stay below `BUDGET_SECONDS`
(default 360 s, i.e. 6 of your 10 minutes). Nothing is submitted until you set `CONFIRM = True`.

**What to send back:** the file `results/hardware/hardware_<backend>_<timestamp>.npz` (raw per-shot data)
plus the CSVs written by the last section.

## 0. Environment

In [1]:
import os, sys, json, time, warnings, pathlib
warnings.filterwarnings("ignore")

REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import numpy as np, pandas as pd
import qiskit, qiskit_ibm_runtime
import qtrace
from qtrace.hardware.runner import HardwareExperiment, load_raw
from qtrace.hardware import analysis as A
print("qiskit", qiskit.__version__, "| qiskit-ibm-runtime", qiskit_ibm_runtime.__version__,
      "| qtrace", qtrace.__version__)
OUT = REPO / "results" / "hardware"; OUT.mkdir(parents=True, exist_ok=True)

qiskit 2.5.2 | qiskit-ibm-runtime 0.49.0 | qtrace 2.0.0


## 1. IBM Quantum account

Run the commented block **once** with your own credentials (never commit them). Afterwards the saved
default account is used automatically.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService

# QiskitRuntimeService.save_account(
    # token="YOUR_API_KEY",
    # instance="YOUR_CRN",
    # overwrite=True,
    # set_as_default=True,
# )

service = QiskitRuntimeService()
backends = service.backends(operational=True, simulator=False)
print(f"Account OK. {len(backends)} backend(s) available:")
for b in backends:
    st = b.status()
    print(f"  {b.name:18s} {b.num_qubits:4d} qubits   pending jobs: {st.pending_jobs}")

Account OK. 3 backend(s) available:
  ibm_fez             156 qubits   pending jobs: 1
  ibm_kingston        156 qubits   pending jobs: 0
  ibm_marrakesh       156 qubits   pending jobs: 0


## 2. Choose the backend and select links

By default the least busy backend with at least 120 qubits is used. Set `BACKEND_NAME` to force one
(e.g. `"ibm_marrakesh"`, `"ibm_fez"`, `"ibm_kingston"`). Pairs are ranked by two-qubit-gate error plus
readout errors and are mutually non-adjacent to suppress crosstalk between links.

In [3]:
BACKEND_NAME = "ibm_marrakesh"          # e.g. "ibm_marrakesh"
N_LINKS      = 40
SEED         = 2026          # fixes the attack schedule and Eve's basis choices

if BACKEND_NAME:
    backend = service.backend(BACKEND_NAME)
else:
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=120)
print("Using", backend.name)

exp = HardwareExperiment(backend, n_links=N_LINKS, n_cal_passes=2, n_epochs=6, seed=SEED)
d = exp.design
print(f"{d.n_links} links selected, {d.n_circuits} circuits "
      f"({d.n_cal_passes} commissioning passes + {d.n_epochs} monitoring epochs)")
pd.DataFrame(exp.link_table).describe().loc[["mean", "min", "max"]]

Using ibm_marrakesh
33 links selected, 64 circuits (2 commissioning passes + 6 monitoring epochs)


,data,ancilla,two_qubit_error,readout_error_data,readout_error_ancilla
mean,79.515152,79.060606,0.002680,0.010317,0.022365
min,2.000000,3.000000,0.001171,0.002930,0.005859
max,153.000000,154.000000,0.005956,0.033081,0.047729


In [4]:
# attack schedule: rows = monitoring epochs, columns = links (-1 = honest)
sched = pd.DataFrame(d.schedule, index=[f"epoch {e}" for e in range(d.n_epochs)])
print("probes:", {i: p.label for i, p in enumerate(d.probes)})
print("link-epochs per probe:", pd.Series(np.array(d.schedule).ravel()).value_counts().sort_index().to_dict())
sched

probes: {0: 'probe(theta=0.150)', 1: 'probe(theta=0.250)', 2: 'probe(theta=0.400)', 3: 'probe(theta=0.600)', 4: 'probe(theta=0.900)', 5: 'probe(theta=1.571)', 6: 'ad(gamma=0.030)', 7: 'ad(gamma=0.060)', 8: 'ad(gamma=0.120)', 9: 'ad(gamma=0.250)'}
link-epochs per probe: {-1: 66, 0: 14, 1: 13, 2: 14, 3: 14, 4: 13, 5: 13, 6: 13, 7: 12, 8: 12, 9: 14}


,0,1,2,3,4,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,32
epoch 0,1,8,-1,-1,2,-1,9,4,-1,-1,...,-1,-1,8,0,1,6,5,-1,1,0
epoch 1,9,3,7,4,-1,6,-1,8,0,7,...,3,7,-1,9,-1,3,2,8,-1,-1
epoch 2,-1,5,4,9,6,-1,1,3,-1,6,...,9,2,3,2,9,-1,9,6,0,6
epoch 3,-1,-1,3,-1,7,5,3,9,1,4,...,5,4,1,-1,0,7,4,3,4,9
epoch 4,0,2,0,1,-1,2,0,-1,2,8,...,6,5,-1,5,8,4,-1,7,2,-1
epoch 5,6,-1,-1,5,8,7,-1,-1,5,-1,...,-1,-1,6,-1,-1,-1,-1,-1,-1,3


In [5]:
t0 = time.time()
isa = exp.transpiled()
print(f"transpiled {len(isa)} circuits in {time.time()-t0:.1f}s; "
      f"max depth {max(c.depth() for c in isa)}; two-qubit gates per circuit "
      f"{min(sum(v for k,v in c.count_ops().items() if k in ('cz','ecr','cx')) for c in isa)}-"
      f"{max(sum(v for k,v in c.count_ops().items() if k in ('cz','ecr','cx')) for c in isa)}")

transpiled 64 circuits in 1.9s; max depth 23; two-qubit gates per circuit 0-62


## 3. Pilot job (≈ a few QPU seconds)

Measures the cost per shot on this backend. Set `CONFIRM = True` to submit.

In [6]:
BUDGET_SECONDS = 360     # pilot + main; keep well below your 600 s allocation
CONFIRM = True

if CONFIRM:
    per_shot, pilot_s = exp.pilot(shots=400)
    print(f"pilot used {pilot_s:.1f} quantum seconds -> {per_shot*1e3:.3f} ms per shot (conservative)")
    SHOTS = exp.plan_shots(per_shot, BUDGET_SECONDS, pilot_s)
    est = SHOTS * d.n_circuits * per_shot
    print(f"planned main job: {d.n_circuits} circuits x {SHOTS} shots, estimated <= {est:.0f} s "
          f"(total <= {est + pilot_s:.0f} s of {BUDGET_SECONDS} s budget)")
else:
    print("Set CONFIRM = True to run the pilot.")

pilot used 3.0 quantum seconds -> 0.938 ms per shot (conservative)
planned main job: 64 circuits x 5900 shots, estimated <= 354 s (total <= 357 s of 360 s budget)


## 4. Main job

Review the plan printed above, then run. The job ID is printed immediately; if the kernel is
interrupted while waiting, use the recovery cell in Section 6.

In [7]:
RUN_MAIN = True

if RUN_MAIN:
    bob, eve = exp.run(SHOTS)
    stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
    RAW = OUT / f"hardware_{backend.name}_{stamp}.npz"
    exp.save(RAW, bob, eve, mode="hardware")
    print("saved raw data ->", RAW)
    print("usage log:", json.dumps(exp.log, indent=1))
else:
    print("Set RUN_MAIN = True to submit the main job.")

submitted job dc0p39ts2omc7389kd30 (64 circuits x 5900 shots); waiting...
saved raw data -> /home/sumit/Q-TRACE-final/Q-TRACE/results/hardware/hardware_ibm_marrakesh_20261121T124423Z.npz
usage log: [
 {
  "stage": "pilot",
  "job_id": "dc0p205hdvas73bjr3j0",
  "shots": 400,
  "circuits": 8,
  "quantum_seconds": 3.0
 },
 {
  "stage": "main",
  "job_id": "dc0p39ts2omc7389kd30",
  "shots": 5900,
  "circuits": 64,
  "quantum_seconds": 104.0,
  "wall_seconds": 132.22055172920227
 }
]


## 5. Analysis (runs locally, no QPU time)

In [8]:
RAW = sorted(OUT.glob("hardware_*.npz"))[-1]          # most recent hardware file
bob, eve, design, meta = load_raw(RAW)
print(RAW.name, "| backend", meta["backend"], "| shots/circuit", bob.shape[1], "| links", bob.shape[2])

n1, n = A.count_tables(bob, design)
tests, disp = A.link_epoch_tests(n1, n, design)
print("dispersion:", {k: round(v, 3) for k, v in disp.items()})
det = A.detection_table(tests)
det

hardware_ibm_marrakesh_20261121T124423Z.npz | backend ibm_marrakesh | shots/circuit 5900 | links 33
dispersion: {'phi': 1.0, 'phi_qber': 1.014, 'phi_qber_basis': 1.026}


,probe,kind,strength,ideal_qber,n,measured_qber_excess,Q-TRACE@0.001,Q-TRACE@0.01,QBER (pooled)@0.001,QBER (pooled)@0.01,QBER (per basis)@0.001,QBER (per basis)@0.01,Omnibus only@0.001,Omnibus only@0.01
9,ad(gamma=0.030),ad,0.030000,0.011279,13,0.040583,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
10,ad(gamma=0.060),ad,0.060000,0.022616,12,0.044686,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,ad(gamma=0.120),ad,0.120000,0.045479,12,0.071487,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
4,ad(gamma=0.250),ad,0.250000,0.095994,14,0.110628,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2,honest,honest,0.000000,0.000000,66,0.010889,0.924242,0.939394,0.924242,0.954545,0.954545,0.969697,0.909091,0.924242
7,probe(theta=0.150),probe,0.150000,0.002807,14,0.014831,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
0,probe(theta=0.250),probe,0.250000,0.007772,13,0.020556,1.000000,1.000000,1.000000,1.000000,0.923077,1.000000,1.000000,1.000000
3,probe(theta=0.400),probe,0.400000,0.019735,14,0.030987,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
6,probe(theta=0.600),probe,0.600000,0.043666,14,0.055337,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
5,probe(theta=0.900),probe,0.900000,0.094598,13,0.101322,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [9]:
curve = A.detection_vs_rounds(bob, design, alpha=1e-3)
r2d = A.rounds_to_detect(curve)
info = A.eve_information(bob, eve, design)
leak = A.leaked_before_detection(info, r2d)
pred = A.predicted_power(bob, design, sorted(curve.shots.unique()))

tag = RAW.stem
tests.to_csv(OUT / f"{tag}_link_epoch_tests.csv", index=False)
det.to_csv(OUT / f"{tag}_detection.csv", index=False)
curve.to_csv(OUT / f"{tag}_detection_vs_rounds.csv", index=False)
pred.to_csv(OUT / f"{tag}_predicted_power.csv", index=False)
info.to_csv(OUT / f"{tag}_eve_information.csv", index=False)
leak.to_csv(OUT / f"{tag}_leakage_before_alarm.csv", index=False)
json.dump(disp, open(OUT / f"{tag}_dispersion.json", "w"), indent=1)
print("written:", sorted(p.name for p in OUT.glob(tag + "_*")))
info

written: ['hardware_ibm_marrakesh_20261121T124423Z_detection.csv', 'hardware_ibm_marrakesh_20261121T124423Z_detection_vs_rounds.csv', 'hardware_ibm_marrakesh_20261121T124423Z_dispersion.json', 'hardware_ibm_marrakesh_20261121T124423Z_eve_information.csv', 'hardware_ibm_marrakesh_20261121T124423Z_leakage_before_alarm.csv', 'hardware_ibm_marrakesh_20261121T124423Z_link_epoch_tests.csv', 'hardware_ibm_marrakesh_20261121T124423Z_predicted_power.csv']


,probe,kind,strength,I_matched,I_sifted,guess_error,I_matched_ideal,I_sifted_ideal
5,ad(gamma=0.030),ad,0.030000,0.003595,0.001797,0.484420,0.015165,0.007582
9,ad(gamma=0.060),ad,0.060000,0.011336,0.005668,0.471215,0.030669,0.015335
8,ad(gamma=0.120),ad,0.120000,0.019408,0.009704,0.450360,0.062764,0.031382
7,ad(gamma=0.250),ad,0.250000,0.082810,0.041405,0.386380,0.137925,0.068963
6,probe(theta=0.150),probe,0.150000,0.014552,0.007276,0.424292,0.016169,0.008085
0,probe(theta=0.250),probe,0.250000,0.028880,0.014440,0.399381,0.044615,0.022307
1,probe(theta=0.400),probe,0.400000,0.090037,0.045018,0.323971,0.112337,0.056169
3,probe(theta=0.600),probe,0.600000,0.181773,0.090886,0.254546,0.244094,0.122047
2,probe(theta=0.900),probe,0.900000,0.389620,0.194810,0.147849,0.505122,0.252561
4,probe(theta=1.571),probe,1.570796,0.698945,0.349473,0.052608,1.000000,0.500000


In [10]:
leak.pivot_table(index="probe", columns="test", values=["shots_to_detect", "bits_leaked_before_alarm"])

bits_leaked_before_alarm                              \
test                           Omnibus only    Q-TRACE QBER (per basis)   
probe                                                                     
ad(gamma=0.030)                    0.447127   0.359470         0.359470   
ad(gamma=0.060)                    1.133585   1.133585         1.133585   
ad(gamma=0.120)                    1.940771   1.940771         1.940771   
ad(gamma=0.250)                    8.281049   8.281049         8.281049   
probe(theta=0.150)                 6.493859   3.896649         3.757055   
probe(theta=0.250)                 7.925900   4.651039         4.084199   
probe(theta=0.400)                11.821761   9.003687         9.003687   
probe(theta=0.600)                18.177251  18.177251        18.177251   
probe(theta=0.900)                38.961984  38.961984        38.961984   
probe(theta=1.571)                69.894527  69.894527        69.894527   

                                 shots_to_detect                               \
test               QBER (pooled)    Omnibus only     Q-TRACE QBER (per basis)   
probe                                                                           
ad(gamma=0.030)         0.359470       62.192599   50.000000        50.000000   
ad(gamma=0.060)         1.133585       50.000000   50.000000        50.000000   
ad(gamma=0.120)         1.940771       50.000000   50.000000        50.000000   
ad(gamma=0.250)         8.281049       50.000000   50.000000        50.000000   
probe(theta=0.150)      3.362454      223.131584  133.890410       129.093920   
probe(theta=0.250)      3.921021      137.222945   80.524517        70.710678   
probe(theta=0.400)      9.003687       65.649555   50.000000        50.000000   
probe(theta=0.600)     18.177251       50.000000   50.000000        50.000000   
probe(theta=0.900)     38.961984       50.000000   50.000000        50.000000   
probe(theta=1.571)     69.894527       50.000000   50.000000        50.000000   

                                  
test               QBER (pooled)  
probe                             
ad(gamma=0.030)        50.000000  
ad(gamma=0.060)        50.000000  
ad(gamma=0.120)        50.000000  
ad(gamma=0.250)        50.000000  
probe(theta=0.150)    115.535270  
probe(theta=0.250)     67.885552  
probe(theta=0.400)     50.000000  
probe(theta=0.600)     50.000000  
probe(theta=0.900)     50.000000  
probe(theta=1.571)     50.000000

## 6. Recovery (only if the kernel stopped while waiting for the main job)

Paste the job ID printed in Section 4. The experiment object must be rebuilt with the **same**
`BACKEND_NAME`, `N_LINKS` and `SEED` (re-run Section 2 first).

In [12]:
JOB_ID = "dc0p39ts2omc7389kd30"   # e.g. "d1abc23xyz..."
if JOB_ID:
    from qtrace.hardware.runner import extract
    job = service.job(JOB_ID)
    bob, eve = extract(job.result(), exp.design.n_links)
    exp.log.append(dict(stage="main(recovered)", job_id=JOB_ID,
                        quantum_seconds=qtrace.hardware.runner.job_quantum_seconds(job)))
    stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
    RAW = OUT / f"hardware_{backend.name}_{stamp}.npz"
    exp.save(RAW, bob, eve, mode="hardware")
    print("saved", RAW)

saved /home/sumit/Q-TRACE-final/Q-TRACE/results/hardware/hardware_ibm_marrakesh_20261121T124536Z.npz
